# Tarea 1 — Inteligencia Artificial (CIT-2013) — Parte 2

> 🤖 **Sugerencia de Claude — no es tu contenido; revisa y decide si la integras o la descartas.**

## [Nombre del ramo y universidad]

- Nombres completos de los integrantes, fecha, URL del dataset *Appliances Energy Prediction* con su tamaño

## Declaración de uso de herramientas de IA generativa

> 🤖 **Sugerencia de Claude — no es tu contenido; revisa y decide si la integras o la descartas.**


- Propósitos específicos + qué no se delegó (diseño, criterios de discretización, análisis de resultados)

## Parte 2 — Cadenas de Markov y Modelos Ocultos de Markov

> 🤖 **Sugerencia de Claude — no es tu contenido; revisa y decide si la integras o la descartas.**


### 2.0 Dataset y esquema del modelo

- Dataset fijo, frecuencia de muestreo (10 min → 144 registros/día), rango temporal
- Esquema: 6 estados ocultos de consumo, 27 observaciones (3×3×3: `lights` × dos índices)
- Las tres piezas del modelo y sus dimensiones: inicial (6), transición (6×6), emisión (6×27)
- Enlace con la Parte 1: un HMM es una red bayesiana con topología de cadena

In [ ]:
# 🤖 Sugerencia de Claude — celda placeholder, aún no es código tuyo.
# C10 · Carga + exploración inicial.

# df = pd.read_csv("energydata_complete.csv", parse_dates=["date"])
# print(df.shape)
# print(df.dtypes)
# print(df["date"].min(), df["date"].max())
# print(df.isna().sum())

### a) Discretización de variables y partición de los datos

> 🤖 **Sugerencia de Claude — no es tu contenido; revisa y decide si la integras o la descartas.**


#### Estados ocultos
Los 6 cortes de `Appliances` (cuantiles o umbrales en watts), justificados, con tabla de rangos.

#### Observaciones
3 niveles de `lights` y de cada índice ($I_{temperatura}$, $I_{humedad}$), combinados en un entero 0–26.

#### Partición
Por qué es cronológica y no aleatoria, y por qué los cortes se calculan solo sobre train.

In [ ]:
# 🤖 Sugerencia de Claude — celda placeholder, aún no es código tuyo.
# C11 · Discretizar + codificar + split (train/test cronológico).

# df["I_temperatura"] = df[["T1","T2","T3"]].mean(axis=1)
# df["I_humedad"] = df[["RH1","RH2","RH3"]].mean(axis=1)

# n_train = int(len(df) * 0.8)   # o un corte por fecha
# train, test = df.iloc[:n_train], df.iloc[n_train:]   # split CRONOLOGICO, sin shuffle

# Cortes calculados SOLO sobre train, luego aplicados a train y test:
# _, edges_appliances = pd.cut(train["Appliances"], bins=6, retbins=True)
# train["estado"] = pd.cut(train["Appliances"], bins=edges_appliances, labels=range(6))
# test["estado"]  = pd.cut(test["Appliances"],  bins=edges_appliances, labels=range(6))

# lights (3 niveles) + I_temperatura (3 niveles) + I_humedad (3 niveles) -> entero 0..26
# codigo_obs = n_lights * 9 + n_temp * 3 + n_hum

### b) Cadenas de Markov: modelo continuo y modelo diario

> 🤖 **Sugerencia de Claude — no es tu contenido; revisa y decide si la integras o la descartas.**


- Propiedad de Markov, matriz de transición (filas suman 1)
- Estimación por conteo de pares consecutivos
- Diferencia entre modelo continuo (cuenta transición nocturna) y diario (no la cuenta, usa `lengths`)
- Distribución inicial: estacionaria (continuo) vs. frecuencia empírica de medianoche (diario)

In [ ]:
# 🤖 Sugerencia de Claude — celda placeholder, aún no es código tuyo.
# C12 · Transiciones + distribuciones iniciales.

# def matriz_transicion(secuencias, K):
#     C = np.zeros((K, K))
#     for s in secuencias:
#         for a, b in zip(s[:-1], s[1:]):
#             C[a, b] += 1
#     filas = C.sum(axis=1, keepdims=True)
#     filas[filas == 0] = 1
#     return C / filas

# T_continuo = matriz_transicion([estados_train], K=6)                 # SI cuenta transicion nocturna
# T_diario   = matriz_transicion(secuencias_por_dia, K=6)              # NO la cuenta
# assert np.allclose(T_continuo.sum(axis=1), 1)
# assert np.allclose(T_diario.sum(axis=1), 1)

# inicial_diaria = frecuencia_del_primer_estado_de_cada_dia(...)      # empirica

### c) Distribución estacionaria y predicción a k pasos

> 🤖 **Sugerencia de Claude — no es tu contenido; revisa y decide si la integras o la descartas.**


- Definición de $\pi$ ($\pi T = \pi$), verificación de irreducibilidad/aperiodicidad
- Interpretación en el dominio, contraste inicial diaria vs. estacionaria
- Predicción a k pasos ($T^k$), k = 6 → una hora
- Al menos 2 consultas de horizonte (continuo) + 2 sobre evolución de una jornada (diario)

In [ ]:
# 🤖 Sugerencia de Claude — celda placeholder, aún no es código tuyo.
# C13 · Distribución estacionaria + potencias de T.

# vals, vecs = np.linalg.eig(T_continuo.T)
# pi = np.real(vecs[:, np.argmin(np.abs(vals - 1))])
# pi = np.abs(pi) / np.abs(pi).sum()
# assert np.allclose(pi @ T_continuo, pi)

# T6 = np.linalg.matrix_power(T_continuo, 6)   # transicion a 6 pasos = 1 hora

### d) Matriz de emisión y construcción de los HMM

> 🤖 **Sugerencia de Claude — no es tu contenido; revisa y decide si la integras o la descartas.**


- $P(\text{observación} \mid \text{estado})$, filas suman 1, estimada por conteo (caso supervisado)
- Por qué NO se usa Baum-Welch / `model.fit()` de hmmlearn

#### Por qué ambos modelos pueden compartir la misma matriz de emisión
La emisión es sincrónica; la segmentación en días solo afecta a los pares consecutivos de transición.

In [ ]:
# 🤖 Sugerencia de Claude — celda placeholder, aún no es código tuyo.
# C14 · Matriz de emisión + construir los HMM.

# from hmmlearn import hmm

# def matriz_emision(estados, obs, K, M):
#     C = np.zeros((K, M))
#     for e, o in zip(estados, obs):
#         C[e, o] += 1
#     return C / C.sum(axis=1, keepdims=True)

# E = matriz_emision(estados_train, obs_train, K=6, M=27)

# hmm_continuo = hmm.CategoricalHMM(n_components=6, init_params="", params="")
# hmm_continuo.startprob_, hmm_continuo.transmat_, hmm_continuo.emissionprob_ = pi, T_continuo, E
# hmm_continuo.n_features = 27

# hmm_diario = hmm.CategoricalHMM(n_components=6, init_params="", params="")
# hmm_diario.startprob_, hmm_diario.transmat_, hmm_diario.emissionprob_ = inicial_diaria, T_diario, E
# hmm_diario.n_features = 27

### e) Inferencias: Forward, Forward-Backward y Viterbi

> 🤖 **Sugerencia de Claude — no es tu contenido; revisa y decide si la integras o la descartas.**


| Algoritmo | Pregunta que responde | Forma de la salida |
|---|---|---|
| Forward | | |
| Forward-Backward | | |
| Viterbi | | |

Qué día de evaluación se usa y por qué.

In [ ]:
# 🤖 Sugerencia de Claude — celda placeholder, aún no es código tuyo.
# C15 · Forward / Forward-Backward / Viterbi sobre un día de test.

# obs_dia = np.array([[o] for o in observaciones_de_un_dia_de_test])

# logprob_continuo = hmm_continuo.score(obs_dia)          # Forward -> log P(observaciones)
# logprob_diario    = hmm_diario.score(obs_dia)

# gamma_continuo = hmm_continuo.predict_proba(obs_dia)    # Forward-Backward -> P(estado_t | todas las obs)

# _, camino_continuo = hmm_continuo.decode(obs_dia, algorithm="viterbi")  # Viterbi -> secuencia mas probable
# _, camino_diario    = hmm_diario.decode(obs_dia, algorithm="viterbi")

# acc_continuo = (camino_continuo == estados_reales_del_dia).mean()
# acc_diario    = (camino_diario == estados_reales_del_dia).mean()

### f) Análisis comparativo de los modelos

> 🤖 **Sugerencia de Claude — no es tu contenido; revisa y decide si la integras o la descartas.**


1. Efecto de incluir/excluir transiciones nocturnas
2. Qué modelo predice mejor y por qué
3. Forward-Backward vs. Viterbi: dónde difieren y por qué
4. Limitaciones comunes (no-estacionariedad real del consumo)

## Conclusiones y limitaciones

> 🤖 **Sugerencia de Claude — no es tu contenido; revisa y decide si la integras o la descartas.**


### Conclusiones
### Limitaciones

## Referencias

> 🤖 **Sugerencia de Claude — no es tu contenido; revisa y decide si la integras o la descartas.**


- Russell & Norvig, *AIMA*, 4ª ed., capítulos 13 y 14
- URL del dataset, con su autor
- Documentación de `hmmlearn`, con la versión utilizada